In [ ]:
#import needed libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import re

#read in the data
fakeData = pd.read_csv("Fake.csv")
trueData = pd.read_csv("True.csv")

#add labels that are standard for AI models. indicated 0 for fake news and 1 for true news
fakeData['label'] = 0
trueData['label'] = 1

#the dataset includes clear indicators of true value so get rid of that string for a more advanced ai model
def cleanData(article):
    #split the article when it finds the string '(Reuters) -'
    identified = article.split('(Reuters) - ', 1)
    if len(identified) > 1:
        #return the text that comes after (Reuters)
        return identified[1]
    #if it's not in the test, just return the entire article
    return article

#update the true csv file 
trueData['identified'] = trueData['text'].apply(cleanData)
fakeData['identified'] = fakeData['text']

#only extract 500 lines of the datasets
useFake = fakeData.sample(n=250, random_state=42)
useTrue = trueData.sample(n=250, random_state=42)

#combine the datasets, randomize the order, delete the index numbers
combineFT = pd.concat([useFake, useTrue])
randomFT = combineFT.sample(frac=1, random_state=42)
finalData = randomFT.reset_index(drop=True)

#split train and test data
trainData, testData, trainLabel, testLabel = train_test_split(
    finalData['identified'].tolist(), finalData['label'].tolist(), test_size=0.2, random_state=42
)

print(f"Train: {len(trainData)}, Test: {len(testData)}")

Train: 240, Test: 60


In [4]:
#the naive baseline
from sklearn.metrics import classification_report

def baseline(article):
    #standard for baseline is the number of capital letters in the title of the article
    upperCount = sum(1 for c in article if c.isupper())
    length = len(article)

    #if more than 5% of the article is uppercase, predict that it is fake news
    if length > 0 and (upperCount / length) > 0.05:
        return 0
    
    return 1

baselineResult = [baseline(str(t)) for t in testData]

print("Naive Baseline Results")
print(classification_report(testLabel, baselineResult, target_names=['Fake', 'True']))

Naive Baseline Results
              precision    recall  f1-score   support

        Fake       0.56      0.19      0.29        26
        True       0.59      0.88      0.71        34

    accuracy                           0.58        60
   macro avg       0.57      0.54      0.50        60
weighted avg       0.57      0.58      0.52        60



In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

#setup the distilBERT ai pipeline model
modelName = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(modelName)
model = DistilBertForSequenceClassification.from_pretrained(modelName, num_labels=2)

#change into strings to solve TypeError
trainDataList = [str(x) for x in trainData]
testDataList = [str(x) for x in testData]

#tokenize the data
trainToken = tokenizer(trainDataList, truncation=True, padding=True, max_length=128)
testToken = tokenizer(testDataList, truncation=True, padding=True, max_length=128)

#create an object for dataset pytorch
class newsData(Dataset):
    def __init__(self, tokenized, label):
        self.tokenized = tokenized
        self.label = label

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        #get calculated numbers from tokenizer and add to the label
        item = {key: torch.tensor(val[idx]) for key, val in self.tokenized.items()}
        item['labels'] = torch.tensor(self.label[idx])
        return item

trainDataset = newsData(trainToken, trainLabel)
testDataset = newsData(testToken, testLabel)

#the actual training process
trainArgs = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=trainArgs,
    train_dataset=trainDataset,
    eval_dataset=testDataset
)

trainer.train()


c:\Users\seulp\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\seulp\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=100, training_loss=0.054445018768310545, metrics={'train_runtime': 1123.4313, 'train_samples_per_second': 1.424, 'train_steps_per_second': 0.089, 'total_flos': 52986959462400.0, 'train_loss': 0.054445018768310545, 'epoch': 2.0})

In [1]:
import numpy as np

#get the predicted results
predResults = trainer.predict(testDataset)
aiPred = np.argmax(predResults.predictions, axis=1)

print("\n AI Model Pipeline Results")
print(classification_report(testLabel, aiPred, target_names=['Fake', 'True']))

#counter for the loop
count = 0

#get the comparisons
print("\n Baseline vs AI model pipeline")
for l in range(len(testLabel)):
    if baselineResult[l] != testLabel[l] and aiPred[l] == testLabel[l]:
        print(f"Index: {l}")
        print(f"True or Fake: {testLabel[l]}")
        print(f"Baseline Prediction: {baselineResult[l]}")
        print(f"AI Prediction: {aiPred[l]}")
        print(f"Tested Article: {str(testData[l])[:100]}...")
        print("\n")

        count += 1
        if count >= 3:
            break

NameError: name 'trainer' is not defined